# Table 5: IRI, Rut Prediction


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    RepeatedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error

from xgboost import XGBRegressor


# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------

RNG = 42
INPUT_FILE = "climate_merged_ML_data.xlsx"

IRI_TARGET = "Mean IRI"
RUT_TARGET = "depth_1_8"


# ------------------------------------------------------------------
# Load and prepare data
# ------------------------------------------------------------------

df = pd.read_excel(INPUT_FILE)

# Remove location because it is not used as a predictor
df = df.drop(columns=["location"])

# Convert pavement-family categories to numeric labels
label_encoder = LabelEncoder()

df["Pavement Family"] = label_encoder.fit_transform(
    df["Pavement Family"].astype(str)
)


# ------------------------------------------------------------------
# Selected model configurations
# ------------------------------------------------------------------

models = {
    "Linear": LinearRegression(),

    "SVR": SVR(
        C=10,
        gamma="scale",
        epsilon=0.1,
    ),

    "KNN": KNeighborsRegressor(
        n_neighbors=25,
        weights="distance",
        p=1,
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        max_features=0.5,
        random_state=RNG,
        n_jobs=-1,
    ),

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        max_features=0.5,
        random_state=RNG,
        n_jobs=-1,
    ),

    "HistGB": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=400,
        l2_regularization=1,
        random_state=RNG,
    ),

    "XGBoost": XGBRegressor(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        reg_lambda=1,
        random_state=RNG,
        n_jobs=-1,
        tree_method="hist",
        objective="reg:squarederror",
    ),
}


# ------------------------------------------------------------------
# Repeated cross-validation
# ------------------------------------------------------------------

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=RNG,
)


# ------------------------------------------------------------------
# Model evaluation function
# ------------------------------------------------------------------

def metrics(estimator, X, y):
    """
    Calculate:
        - Training R2
        - Mean repeated cross-validation R2
        - Test R2
        - Test MAE
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RNG,
    )

    model_pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", clone(estimator)),
        ]
    )

    # Fit model on training data
    model_pipeline.fit(X_train, y_train)

    # Predictions
    train_predictions = model_pipeline.predict(X_train)
    test_predictions = model_pipeline.predict(X_test)

    # Training-set R2
    train_r2 = r2_score(
        y_train,
        train_predictions,
    )

    # Hold-out test-set R2
    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    # Hold-out test-set MAE
    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    # Repeated cross-validation R2
    cv_pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", clone(estimator)),
        ]
    )

    cv_scores = cross_val_score(
        cv_pipeline,
        X,
        y,
        cv=rkf,
        scoring="r2",
        n_jobs=-1,
    )

    cv_r2 = cv_scores.mean()

    return (
        round(train_r2, 3),
        round(cv_r2, 3),
        round(test_r2, 3),
        round(test_mae, 3),
    )


# ------------------------------------------------------------------
# Define predictor and target datasets
# ------------------------------------------------------------------

predictor_columns = [
    column
    for column in df.columns
    if column not in [IRI_TARGET, RUT_TARGET]
]

X_iri = df[predictor_columns]
y_iri = df[IRI_TARGET]

X_rut = df[predictor_columns]
y_rut = df[RUT_TARGET]


# ------------------------------------------------------------------
# Evaluate all models
# ------------------------------------------------------------------

rows = []

for model_name, estimator in models.items():

    iri_train_r2, iri_cv_r2, iri_test_r2, iri_mae = metrics(
        estimator,
        X_iri,
        y_iri,
    )

    rut_train_r2, rut_cv_r2, rut_test_r2, rut_mae = metrics(
        estimator,
        X_rut,
        y_rut,
    )

    rows.append(
        [
            model_name,
            iri_train_r2,
            iri_cv_r2,
            iri_test_r2,
            iri_mae,
            rut_train_r2,
            rut_cv_r2,
            rut_test_r2,
            rut_mae,
        ]
    )


# ------------------------------------------------------------------
# Create results table
# ------------------------------------------------------------------

tab = pd.DataFrame(
    rows,
    columns=[
        "Model",
        "IRI_Train_R2",
        "IRI_CV_R2",
        "IRI_Test_R2",
        "IRI_MAE",
        "Rut_Train_R2",
        "Rut_CV_R2",
        "Rut_Test_R2",
        "Rut_MAE",
    ],
)


# ------------------------------------------------------------------
# Display results
# ------------------------------------------------------------------

print(
    "\nTable 4. Condition-model performance "
    "(structure + traffic + climate)\n"
)

print(tab.to_string(index=False))


# ------------------------------------------------------------------
# Optional: save the results to Excel
# ------------------------------------------------------------------

tab.to_excel(
    "Table_4_condition_model_performance.xlsx",
    index=False,
)

print(
    "\nResults saved as: "
    "Table_4_condition_model_performance.xlsx"
)


Table 4. Condition-model performance (structure + traffic + climate)

       Model  IRI_Train_R2  IRI_CV_R2  IRI_Test_R2  IRI_MAE  Rut_Train_R2  Rut_CV_R2  Rut_Test_R2  Rut_MAE
      Linear         0.237      0.238        0.309    0.341         0.251      0.228        0.223    1.859
         SVR         0.657      0.393        0.412    0.293         0.521      0.390        0.381    1.478
         KNN         0.996      0.493        0.498    0.280         0.998      0.507        0.570    1.383
RandomForest         0.869      0.539        0.575    0.259         0.873      0.571        0.638    1.227
  ExtraTrees         0.878      0.545        0.567    0.257         0.875      0.573        0.632    1.227
      HistGB         0.928      0.513        0.535    0.270         0.936      0.552        0.626    1.206
     XGBoost         0.871      0.524        0.530    0.266         0.884      0.547        0.617    1.230

Results saved as: Table_4_condition_model_performance.xlsx


# SHAP IRI and RUT

In [ ]:
# ==================================================================
# Full SHAP importance tables (all features) for the condition models,
# plus the category aggregation that produces the pie-chart percentages.
# Input: climate_merged_ML_data.xlsx   Needs: pandas, scikit-learn, shap
# ==================================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesRegressor
import shap

RNG = 42
df = pd.read_excel("climate_merged_ML_data.xlsx").drop(columns=["location"])
df["Pavement Family"] = LabelEncoder().fit_transform(df["Pavement Family"])
targets = ["Mean IRI", "depth_1_8"]
rf = dict(n_estimators=400, max_depth=None, min_samples_leaf=2, max_features=0.5, random_state=RNG, n_jobs=-1)

CLIMATE = {"MAX_ANN_HUM_AVG","MIN_ANN_HUM_AVG","TOTAL_ANN_PRECIP","WET_DAYS_YR",
           "TOTAL_SNOWFALL_YR","MEAN_ANN_TEMP_AVG","MAX_ANN_TEMP","FREEZE_THAW_YR"}
TRAFFIC = {"annual traffic","KESAL_YEAR"}
def category(f):
    if f in CLIMATE: return "Climate"
    if f in TRAFFIC: return "Traffic"
    return "Structural"

def full_table(tgt):
    other = [t for t in targets if t != tgt][0]
    X = df.drop(columns=[tgt, other]); y = df[tgt]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RNG)
    model = ExtraTreesRegressor(**rf).fit(Xtr, ytr)
    sv = shap.TreeExplainer(model).shap_values(Xte)
    phi = np.abs(sv).mean(axis=0)                 # mean|SHAP| per feature
    rel = phi / phi.sum() * 100                   # relative importance %
    t = pd.DataFrame({"Feature": X.columns, "mean|SHAP|": phi.round(4),
                      "Rel.%": rel.round(2), "Category": [category(c) for c in X.columns]})
    return t.sort_values("Rel.%", ascending=False).reset_index(drop=True)

for tgt, name in [("Mean IRI", "IRI"), ("depth_1_8", "Rut Depth")]:
    t = full_table(tgt)
    print(f"\n================  FULL SHAP TABLE - {name}  ================")
    print(t.to_string(index=False))
    cat = t.groupby("Category")["Rel.%"].sum().reindex(["Structural","Traffic","Climate"])
    print(f"\nCategory totals (sum of Rel.% within each category) - {name}:")
    for k, v in cat.items():
        print(f"  {k:<11} = {v:.1f}%")
    print(f"  ---------------------------\n  TOTAL       = {cat.sum():.1f}%")


================  FULL SHAP TABLE - IRI  ================
                     Feature  mean|SHAP|  Rel.%   Category
                Total_Layers      0.0845  17.08 Structural
             Pavement Family      0.0598  12.09 Structural
          AC layer thickness      0.0450   9.09 Structural
          PC layer thickness      0.0446   9.02 Structural
              annual traffic      0.0248   5.01    Traffic
             MAX_ANN_HUM_AVG      0.0242   4.90    Climate
           MEAN_ANN_TEMP_AVG      0.0218   4.41    Climate
                Base treated      0.0215   4.34 Structural
             MIN_ANN_HUM_AVG      0.0212   4.30    Climate
              FREEZE_THAW_YR      0.0210   4.24    Climate
                MAX_ANN_TEMP      0.0207   4.18    Climate
                  KESAL_YEAR      0.0183   3.70    Traffic
            TOTAL_ANN_PRECIP      0.0181   3.66    Climate
           TOTAL_SNOWFALL_YR      0.0173   3.51    Climate
                 WET_DAYS_YR      0.0151   3.06    Clima

text# When were Rehabilitation efforts taken?
Hint: take the immidiate value before Rehab starts

In [ ]:
import pandas as pd

df = pd.read_excel("ANALYSIS_IRI.xlsx")
df["VISIT_DATE"] = pd.to_datetime(df["VISIT_DATE"])

rows = []
for sid, g in df.groupby("ID"):
    v = g.groupby("VISIT_DATE")["Mean RI"].mean().sort_index().dropna()
    if len(v) < 2:
        continue
    rng = v.max() - v.min()
    if rng <= 0:
        continue
    thr = 0.9 * rng
    vals = v.values
    for i in range(1, len(vals)):
        if vals[i-1] - vals[i] >= thr:
            rows.append((str(sid), round(float(vals[i-1]), 3)))
            break

out = pd.DataFrame(rows, columns=["ID", "Rehab_trigger_m_km"])
out["Rehab_trigger_in_mi"] = (out["Rehab_trigger_m_km"] * 63.36).round(1)
out = out.sort_values("ID").reset_index(drop=True)
print(out)
print(f"Average rehab trigger = {out['Rehab_trigger_m_km'].mean():.2f} ± "
      f"{out['Rehab_trigger_m_km'].std():.2f} m/km  |  "
      f"{out['Rehab_trigger_in_mi'].mean():.1f} ± "
      f"{out['Rehab_trigger_in_mi'].std():.1f} in/mi")

         ID  Rehab_trigger_m_km  Rehab_trigger_in_mi
0    010106               0.747                 47.3
1    010112               0.805                 51.0
2    010162               0.893                 56.6
3    010507               1.245                 78.9
4    010509               1.710                108.3
..      ...                 ...                  ...
626  89A902               3.041                192.7
627  906405               2.270                143.8
628  906420               5.481                347.3
629  90A310               2.873                182.0
630  90B310               1.170                 74.1

[631 rows x 3 columns]
Average rehab trigger = 1.96 ± 0.85 m/km  |  124.0 ± 53.6 in/mi


# Section 6 Result


In [ ]:
# =====================================================================
# Rehabilitation-interval model (Part II) — reproducible pipeline
# Input : rehab_interval_merged_157.xlsx  (157 sections, target = last col)
# Output: Table 7 (MCCV performance) and Figure 9 (SHAP, with category math)
# =====================================================================

# ---------- 0. Install (run once) ----------
# pip install pandas numpy scikit-learn xgboost shap matplotlib openpyxl

# ---------- 1. Imports ----------
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              HistGradientBoostingRegressor)
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import shap

RNG = 0
INPUT = "rehab_interval_merged_157.xlsx"

# ---------- 2. Load data ----------
df = pd.read_excel(INPUT)
df["ID"] = df["ID"].astype(str)         # keep leading zeros
print(f"Loaded {len(df)} sections, {df.shape[1]} columns")

# ---------- 3. Define X / y ----------
# y = 'rehab interval' (last column).
# Excluded from X: ID; the duplicate lower-case 'location' id column;
# and 'Mnt  Rhb count' (the interval's own denominator -> leakage).
y = df["rehab interval"].astype(float)
DROP = ["ID", "location", "Mnt  Rhb count", "rehab interval"]
X = df.drop(columns=DROP)
X["Location"] = X["Location"].astype(str)      # categorical, not numeric

CAT = ["Location", "Base treated", "Engineering Fabric", "Pavement Family"]
NUM = [c for c in X.columns if c not in CAT]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore"))]), CAT),
])

# ---------- 4. Models + Table-2 hyperparameter grids ----------
MODELS = {
 "Linear":        (LinearRegression(), {}),
 "SVR (RBF)":     (SVR(kernel="rbf", gamma="scale"),
                   {"m__C": [10, 30], "m__epsilon": [0.05, 0.1]}),
 "KNN":           (KNeighborsRegressor(weights="distance"),
                   {"m__n_neighbors": [10, 15, 25], "m__p": [1, 2]}),
 "Random Forest": (RandomForestRegressor(n_estimators=400, max_features=0.5,
                   random_state=RNG),
                   {"m__max_depth": [12, None], "m__min_samples_leaf": [2, 4]}),
 "Extra Trees":   (ExtraTreesRegressor(n_estimators=400, max_features=0.5,
                   random_state=RNG),
                   {"m__max_depth": [16, None], "m__min_samples_leaf": [2, 4]}),
 "HistGB":        (HistGradientBoostingRegressor(max_iter=400, random_state=RNG),
                   {"m__learning_rate": [0.05, 0.1], "m__l2_regularization": [1, 5]}),
 "XGBoost":       (XGBRegressor(learning_rate=0.05, subsample=0.8,
                   random_state=RNG, verbosity=0),
                   {"m__n_estimators": [400, 500], "m__max_depth": [3, 4],
                    "m__reg_lambda": [1, 5]}),
}

# ---------- 5. Table 7 : Monte-Carlo CV (10 independent 80/20 splits) ----------
# For each model: 10 random train/test partitions. Hyperparameters are tuned
# by grid search INSIDE each split's training data only (5-fold), so the test
# set never informs fitting or selection. We report the mean Train R2, the mean
# test R2 with its across-split std, and the mean absolute error.
SEEDS   = list(range(1, 11))
TUNE_CV = KFold(n_splits=5, shuffle=True, random_state=RNG)

rows = []
for name, (est, grid) in MODELS.items():
    tr, te, mae = [], [], []
    for s in SEEDS:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=s)
        pipe = Pipeline([("pre", pre), ("m", est)])
        best = (GridSearchCV(pipe, grid, cv=TUNE_CV, scoring="r2", n_jobs=-1)
                .fit(Xtr, ytr).best_estimator_) if grid else pipe.fit(Xtr, ytr)
        tr.append(r2_score(ytr, best.predict(Xtr)))
        p = best.predict(Xte)
        te.append(r2_score(yte, p)); mae.append(mean_absolute_error(yte, p))
    rows.append([name, np.mean(tr), np.mean(te), np.std(te), np.mean(mae)])

T7 = pd.DataFrame(rows, columns=["Model", "Train_R2", "MCCV_Test_R2",
                                 "Test_R2_std", "MCCV_MAE"])
print("\n================ TABLE 7 ================")
print(T7.round(3).to_string(index=False))
best_name = T7.loc[T7.MCCV_Test_R2.idxmax(), "Model"]
print(f"Best model by mean MCCV test R2: {best_name}")

# ---------- 6. Fit best model on one split for SHAP ----------
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=1)
est, grid = MODELS[best_name]
pipe = Pipeline([("pre", pre), ("m", est)])
best = (GridSearchCV(pipe, grid, cv=TUNE_CV, scoring="r2", n_jobs=-1)
        .fit(Xtr, ytr).best_estimator_) if grid else pipe.fit(Xtr, ytr)

pre_fit, model = best.named_steps["pre"], best.named_steps["m"]
Xte_t = pre_fit.transform(Xte)
ohe_names = pre_fit.named_transformers_["cat"].named_steps["oh"].get_feature_names_out(CAT)
feat_names = list(NUM) + list(ohe_names)
Xte_df = pd.DataFrame(Xte_t, columns=feat_names)

# ---------- 7. SHAP: per-feature global importance = mean(|SHAP|) in YEARS ----
sv = shap.TreeExplainer(model).shap_values(Xte_df)   # (n_test, n_feat)
mean_abs = np.abs(sv).mean(axis=0)
imp = pd.DataFrame({"feature": feat_names, "mean_abs_shap": mean_abs})

# collapse one-hot dummies back to their parent variable (sum their importances)
def parent(f):
    for c in CAT:
        if f.startswith(c + "_"):
            return c
    return f
imp["parent"] = imp["feature"].apply(parent)
indiv = imp.groupby("parent")["mean_abs_shap"].sum().sort_values(ascending=False)

# ---------- 8. Category math (traffic / structural / climatic) --------------
# Assign each parent feature to ONE category, then sum member importances.
STRUCTURAL = ["Total_Layers", "Base treated", "Engineering Fabric",
              "Total Base Layer thickness", "Treated Base layer thickness",
              "AC layer thickness", "PC layer thickness", "EF layer thickness",
              "Total_Layer_Thickness", "Pavement Family"]
TRAFFIC    = ["annual traffic (AVG)", "KESAL_YEAR (Avg)", "KESAL_YEAR_RATE",
              "AADTT_ALL_TRUCKS_TREND_RATE", "CMLTV_VOL_HEAVY_TRUCKS_TREND_RATE",
              "IRI_CHANGE_RATE", "RUT_CHANGE_RATE"]
CLIMATIC   = ["Location", "MAX_ANN_HUM_AVG", "MIN_ANN_HUM_AVG", "TOTAL_ANN_PRECIP",
              "WET_DAYS_YR", "TOTAL_SNOWFALL_YR", "MEAN_ANN_TEMP_AVG",
              "MAX_ANN_TEMP", "FREEZE_THAW_YR"]
def category(p):
    if p in STRUCTURAL: return "Structural"
    if p in TRAFFIC:    return "Traffic"
    if p in CLIMATIC:   return "Climatic"
    return "Other"

grp = indiv.reset_index(); grp["category"] = grp["parent"].apply(category)
cat_val = grp.groupby("category")["mean_abs_shap"].sum().sort_values(ascending=False)
cat_pct = (cat_val / cat_val.sum() * 100).round(1)

print("\n=========== SHAP CATEGORY AGGREGATION ===========")
print("Sum of mean|SHAP| within each category (years):")
print(cat_val.round(3).to_string())
print("\nShare of total attribution (%):")
print(cat_pct.to_string())
print("\nTop individual features (mean|SHAP|, years):")
print(indiv.head(8).round(3).to_string())



Loaded 157 sections, 30 columns

================ TABLE 7 ================
        Model  Train_R2  MCCV_Test_R2  Test_R2_std  MCCV_MAE
       Linear     0.796         0.241        0.268     3.091
    SVR (RBF)     0.852         0.473        0.127     2.428
          KNN     1.000         0.440        0.100     2.843
Random Forest     0.902         0.558        0.117     2.372
  Extra Trees     0.948         0.553        0.133     2.245
       HistGB     0.993         0.631        0.155     2.159
      XGBoost     0.999         0.615        0.137     2.085
Best model by mean MCCV test R2: HistGB

=========== SHAP CATEGORY AGGREGATION ===========
Sum of mean|SHAP| within each category (years):
category
Traffic       4.329
Climatic      2.932
Structural    2.719

Share of total attribution (%):
category
Traffic       43.4
Climatic      29.4
Structural    27.2

Top individual features (mean|SHAP|, years):
parent
CMLTV_VOL_HEAVY_TRUCKS_TREND_RATE    1.546
Total Base Layer thickness        

## Files mentioned for import:

*   `climate_merged_ML_data.xlsx`
*   `ANALYSIS_IRI.xlsx`
*   `rehab_interval_merged_157.xlsx`